In [277]:


import jax 
import jax.numpy as jnp


import haiku as hk
import optax

from probjax.nn.transformers import Transformer
from probjax.nn.tokenizer import scalarize, ScalarTokenizer
from probjax.nn.helpers import GaussianFourierEmbedding
from probjax.nn.loss_fn import denoising_score_matching_loss

from probjax.distributions.sde import VPSDE, BaseSDE
from probjax.distributions import Normal, Independent
from probjax.distributions.transformed_distribution import TransformedDistribution
from probjax.distributions.discrete import Empirical

from probjax.utils.sdeint import sdeint

from functools import partial




In [342]:
from sbibm import get_task

task = get_task("slcp")
prior = task.get_prior_dist()
simulator = task.get_simulator()

thetas = prior.sample((100000,))
xs = simulator(thetas)

thetas = jnp.array(thetas)
xs = jnp.array(xs)


In [343]:


def conditional_mlp(output_dim:int, hidden_dim:int = 100, num_hidden:int=8, activation=jax.nn.gelu, layer_norm:bool = True, output_scale_fn=None):
    
    if output_scale_fn is None:
        output_scale_fn = lambda t, x: x
    
    def score_net(t, x, context):
        
        #x, context = jnp.broadcast_arrays(x, context)
        x = jnp.concatenate([x, context], axis=-1)
        
        time_embedding = GaussianFourierEmbedding(hidden_dim)(t[...,None])
        h = activation(hk.Linear(hidden_dim)(x) + time_embedding)
        
        
        for _ in range(num_hidden - 1):
            h_new = hk.Linear(hidden_dim)(h)
            h_new += time_embedding
            h = activation(h_new)
            
            if layer_norm:
                h = hk.LayerNorm(axis=-1, create_scale=True, create_offset=True)(h)
            
        out = hk.Linear(output_dim)(h)
        out = output_scale_fn(t, out)
        return out
    
    init_fn, apply_fn = hk.without_apply_rng(hk.transform(score_net))
    return init_fn, apply_fn

def init_sde_related(data, name="vpsde", **kwargs):
    # VPSDE 
    if name.lower() == "vpsde":
        p0 = Independent(Empirical(data), 1)
        beta_max = kwargs.get("beta_max",10.)
        beta_min = kwargs.get("beta_min", 0.01)
        sde = VPSDE(p0, beta_max=beta_max, beta_min=beta_min)
        T_max = kwargs.get("T_max", 1.)
        T_min = kwargs.get("T_min", 1e-5)

        # Train weight function
        def weight_fn(t):
            t = t.reshape(-1, 1)
            return jnp.clip(1-jnp.exp(-0.5 * (beta_max - beta_min) * t**2 - beta_min * t) ,a_min = 1e-4)
        
        # Model output scale function
        def output_scale_fn(t, x):
            scale = jnp.sqrt(jnp.sum(sde.marginal_variance(t[..., None], x0=jnp.ones_like(x)), -1))
            return jnp.clip(1/scale[..., None],a_min=0.01, a_max=100.) * x
        
    else:
        raise NotImplementedError()
    
    return sde, T_min, T_max, weight_fn, output_scale_fn



def run_train_conditional_score_model(key, params, opt_state, data, num_epochs, num_steps, batch_size_per_device, num_devices,  update, print_every=100):
    # Replicated for multiple devices
    replicated_params = jax.tree_map(lambda x: jnp.array([x] * num_devices), params)
    replicated_opt_state = jax.tree_map(lambda x: jnp.array([x] * num_devices), opt_state)

    for j in range(num_epochs):
        l = 0
        for i in range(num_steps):
            key, key_batch, key_update = jax.random.split(key, 3)
            data_batch = jax.random.choice(key_batch,data, shape=(num_devices, batch_size_per_device, ), axis=0, replace=True)
            loss, replicated_params, replicated_opt_state = update(replicated_params, replicated_opt_state, jax.random.split(key_update, (num_devices,)), data_batch)
            l += loss[0] /num_steps
        if (j % print_every) == 0:     
            print("Train loss: ",l)
            
    params = jax.tree_map(lambda x: x[0], replicated_params)
    
    return params


class NPSE:
    def __init__(self, params, model_fn, sde, T_min=1e-5,T_max=1.) -> None:
        self.params = params
        self.model_fn = model_fn
        self.sde = sde

        self.T_min = T_min
        self.T_max = T_max
        self.marginal_end_std = sde.marginal_stddev(jnp.array([T_max]))
        self.marginal_end_mean = sde.marginal_mean(jnp.array([T_max]))
        
        
        
    def sample(self, key,num_samples, x_o, num_steps=500, **kwargs):
        key1, key2 = jax.random.split(key, 2)
        drift, diffusion = self._init_backward_sde(x_o)
        x_T = jax.random.normal(key1, (num_samples,) + self.sde.event_shape) * self.marginal_end_std + self.marginal_end_mean
        keys = jax.random.split(key2, (num_samples,))
        ys = jax.vmap(lambda *args: sdeint(*args, noise_type="diagonal",**kwargs), in_axes= (0, None, None, 0, None), out_axes=0)(keys, drift, diffusion, x_T, jnp.linspace(0., self.T_max-self.T_min, num_steps))
        return ys[:, -1, ...]
    
    def log_prob(self, val, x_o, **kwargs):
        # Add backward ode to compute log_prob
        raise NotImplementedError()
    
        
    def _init_backward_sde(self, x_o):
        def drift_backward(t, x):
            t = (self.T_max-t)
            score = self.model_fn(self.params, jnp.atleast_1d(t), x, jnp.squeeze(x_o))
            drift = self.sde.drift(t, x)  - self.sde.diffusion(t, x)**2 * score
            return -drift.reshape(x.shape)
        
        def diffusion_backward(t, x):
            t = (self.T_max-t)
            return self.sde.diffusion(t, x).reshape(x.shape)
        
        return drift_backward, diffusion_backward
                 


In [344]:
data = jnp.hstack([thetas, xs])
theta_dim = thetas.shape[-1]
x_dim = xs.shape[-1]

rng = jax.random.PRNGKey(0)

sde, T_min,T_max, weight_fn, output_scale_fn = init_sde_related(thetas)

In [353]:
init_fn, model_fn = conditional_mlp(theta_dim, output_scale_fn=output_scale_fn)
rng, rng_init = jax.random.split(rng)
params = init_fn(rng_init, jnp.ones((10,)), thetas[:10], xs[:10])

In [354]:
total_number_steps = data.shape[0] *2
batch_size = 5000
num_devices = jax.device_count()
batch_size_per_device = batch_size // num_devices
num_steps = data.shape[0] // batch_size + 1

num_epochs = total_number_steps // num_steps 
print_every = num_epochs // 10
learning_rate = 5e-4
schedule = optax.linear_schedule(learning_rate, 0., total_number_steps//2, total_number_steps//2)
optimizer = optax.chain(optax.adaptive_grad_clip(5.), optax.adam(schedule))
opt_state = optimizer.init(params)

In [355]:
@jax.jit
def loss_fn(params, key, data):
    thetas, xs = jnp.split(data, [theta_dim,], axis=-1)
    key_times, key_loss = jax.random.split(key,2)
    times = jax.random.uniform(key_times, (data.shape[0],), minval=T_min, maxval =T_max)
    loss = denoising_score_matching_loss(params, key_loss, times, thetas, None, xs, model_fn = model_fn, mean_fn = sde.marginal_mean, std_fn=sde.marginal_stddev, weight_fn=weight_fn, axis=-1)
    return loss

@partial(jax.pmap, axis_name="num_devices")
def update(params, opt_state, key, data):
    loss, grads = jax.value_and_grad(loss_fn)(params, key, data)

    loss = jax.lax.pmean(loss, axis_name="num_devices")
    grads = jax.lax.pmean(grads, axis_name="num_devices")
    
    updates, opt_state = optimizer.update(grads, opt_state, params=params)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state


In [356]:
rng, rng_train = jax.random.split(rng)
params = run_train_conditional_score_model(rng_train, params, opt_state, data, num_epochs, num_steps, batch_size_per_device, num_devices, update, print_every=print_every)

Train loss:  5.321045
Train loss:  1.695206
Train loss:  1.7293384
Train loss:  1.6658771
Train loss:  1.6424848
Train loss:  1.6500541
Train loss:  1.7200003
Train loss:  1.634878
Train loss:  1.6820518
Train loss:  1.695311
Train loss:  1.6864412


In [357]:
model = NPSE(params, model_fn, sde, T_min, T_max)

In [358]:
from sbi.analysis.sbc import c2st
import torch
import numpy as np

In [359]:

c2sts = []
for i in range(1,11):
    x_o = task.get_observation(i)
    samples_post = task.get_reference_posterior_samples(i)
    x_o = jnp.array(x_o)
    samples_est = model.sample(jax.random.PRNGKey(i), 10000, x_o)
    metric = c2st(torch.tensor(np.array(samples_est)), samples_post)
    print("C2ST: ", metric)

C2ST:  tensor([0.6383])
C2ST:  tensor([0.6154])
C2ST:  tensor([0.6079])


In [ ]:
def train_conditional_score_model(task, thetas,xs, method_cfg, rng):

    device = method_cfg.device
    sde_params = method_cfg.sde_params
    model_params = method_cfg.model_params
    train_params = method_cfg.params_train
    
    
    # Data
    data = jnp.hstack([thetas, xs])
    theta_dim = thetas.shape[-1]
    x_dim = xs.shape[-1]
    
    # Initialize stuff
    sde, T_min,T_max, weight_fn, output_scale_fn = init_sde_related(data, **sde_params)
    init_fn, model_fn = conditional_mlp(theta_dim, output_scale_fn=output_scale_fn, **model_params)
    
    rng, rng_init = jax.random.split(rng)
    params = init_fn(rng_init, jnp.ones((10,)), thetas[:10], xs[:10])

    
    num_epochs = train_params.max_num_epochs
    batch_size = train_params.training_batch_size
    num_devices = jax.device_count()
    batch_size_per_device = batch_size // num_devices
    num_steps = data.shape[0] // batch_size + 1
    
    total_number_steps = num_epochs * num_steps
    learning_rate = train_params.learning_rate
    schedule = optax.linear_schedule(learning_rate, 0., total_number_steps//2, total_number_steps//2)
    optimizer = optax.chain(optax.adaptive_grad_clip(train_params.clip_max_norm), optax.adam(schedule))
    opt_state = optimizer.init(params)
    
    @jax.jit
    def loss_fn(params, key, data):
        thetas, xs = jnp.split(data, 2, axis=-1)
        key_times, key_loss = jax.random.split(key,2)
        times = jax.random.uniform(key_times, (data.shape[0],), minval=T_min, maxval =T_max)
        loss = denoising_score_matching_loss(params, key_loss, times, thetas, None, xs, model_fn = model_fn, mean_fn = sde.marginal_mean, std_fn=sde.marginal_stddev, weight_fn=weight_fn, axis=-1)
        return loss

    @partial(jax.pmap, axis_name="num_devices")
    def update(params, opt_state, key, data):
        loss, grads = jax.value_and_grad(loss_fn)(params, key, data)

        loss = jax.lax.pmean(loss, axis_name="num_devices")
        grads = jax.lax.pmean(grads, axis_name="num_devices")
        
        updates, opt_state = optimizer.update(grads, opt_state, params=params)
        params = optax.apply_updates(params, updates)
        return loss, params, opt_state
    
    
    
    params = run_train_conditional_score_model(rng, params, opt_state, data, num_epochs, num_steps, batch_size_per_device, num_devices, update, print_every=100)
    
    
    marginal_end_std = sde.marginal_stddev(jnp.array([T_max]))[..., : theta_dim]
    marginal_end_mean = sde.marginal_mean(jnp.array([T_max]))[..., : theta_dim]
    
    def init_backward_sde(x_o):
    
        def score_fn(t, theta):
            theta, observed = jnp.broadcast_arrays(theta, x_o)
            score = model_fn(params, jnp.atleast_1d(t), theta, observed)
            return score.reshape(theta.shape)

        def drift_backwards(t, theta):
            t = (1- t)
            score = score_fn(t, theta)
            drift = sde.drift(t, theta) - sde.diffusion(t, theta)**2 * score 
            drift = -drift
            return drift.reshape(theta.shape)

        def diffusion_backwards(t, theta):
            t = (1-t)
            diffusion = sde.diffusion(t, theta) 
            return diffusion

        return drift_backwards, diffusion_backwards
    
    def sample_fn(key, shape, x_o):
        key1, key2 = jax.random.split(key, 2)
        drift, diffusion = init_backward_sde(x_o)
        x_T = jax.random.normal(key1, shape + (theta_dim,)) * marginal_end_std + marginal_end_mean
        keys = jax.random.split(key2, shape)
        ys = jax.vmap(lambda *args: sdeint(*args, noise_type="diagonal"), in_axes= (0, None, None, 0, None), out_axes=0)(keys, drift, diffusion, x_T, jnp.linspace(0., 1-T_min, 500))
        return ys[:, -1, ...]
    
    
    return params, model_fn, sample_fn, sde, T_min, T_max